In [1]:
import cv2
import numpy as np

try:
    from rknnlite.api import RKNNLite as RKNN
except ImportError:
    from rknn.api import RKNN


class YoloOBBRKNN:
    def __init__(self, model_path, input_size=640):
        self.model_path = model_path
        self.input_size = input_size
        self.rknn = RKNN()

    def letterbox(self, im, color=(0, 0, 0)):
        shape = im.shape[:2]
        ratio = min(self.input_size / shape[0], self.input_size / shape[1])
        new_unpad = int(round(shape[1] * ratio)), int(round(shape[0] * ratio))

        if shape[::-1] != new_unpad:
            im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)

        bottom = self.input_size - new_unpad[1]
        right = self.input_size - new_unpad[0]
        im = cv2.copyMakeBorder(
            im, 0, bottom, 0, right, cv2.BORDER_CONSTANT, value=color
        )
        return im, ratio

    def preprocess(self, image_input):
        """
        이미지 경로(str) 또는 이미 로드된 numpy array(BGR)를 입력받아
        NPU 입력용 tensor, ratio, 원본 BGR 이미지를 반환합니다.
        
        Returns:
            input_data (np.ndarray): (1, 640, 640, 3) NPU 입력 데이터
            ratio (float): Letterbox 변환 비율
            orig_img (np.ndarray): 시각화/후처리에 사용할 원본 BGR 이미지
        """
        if isinstance(image_input, str):
            orig_img = cv2.imread(image_input)
        else:
            orig_img = image_input

        img_rgb = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
        img_letterboxed, ratio = self.letterbox(img_rgb)
        
        # Batch 차원 추가: (640, 640, 3) -> (1, 640, 640, 3)
        input_data = np.expand_dims(img_letterboxed, axis=0)

        return input_data, ratio, orig_img

    def decode_outputs(self, outputs, ratio, score_thresh=0.45, nms_thresh=0.25):
        feat_outputs = outputs[:3]
        angle_tensor = outputs[3].squeeze()  # (8400,)

        strides = [8, 16, 32]
        raw_detections = []

        global_box_idx = 0

        for idx, feat in enumerate(feat_outputs):
            stride = strides[idx]
            if feat.ndim == 4:
                feat = feat[0]  # (19, H, W)

            channels, height, width = feat.shape

            cls_probs = feat[4:, :, :]  # (15, H, W)
            max_cls_scores = np.max(cls_probs, axis=0)  # (H, W)
            max_cls_ids = np.argmax(cls_probs, axis=0)    # (H, W)

            mask = max_cls_scores > score_thresh
            ys, xs = np.where(mask)

            box_preds = feat[:4, :, :]  # (4, H, W)

            for y, x in zip(ys, xs):
                score = max_cls_scores[y, x]
                cls_id = max_cls_ids[y, x]

                cx_model = (x + 0.5) * stride
                cy_model = (y + 0.5) * stride

                b_raw = box_preds[:, y, x]
                l, t, r, b = b_raw[0], b_raw[1], b_raw[2], b_raw[3]

                w_model = (l + r) * stride
                h_model = (t + b) * stride

                if w_model <= 0 or h_model <= 0 or w_model > self.input_size * 2:
                    w_model = np.exp(np.clip(r, -10, 10)) * stride
                    h_model = np.exp(np.clip(b, -10, 10)) * stride

                cx_orig = cx_model / ratio
                cy_orig = cy_model / ratio
                w_orig = w_model / ratio
                h_orig = h_model / ratio

                flat_offset = y * width + x
                current_idx = global_box_idx + flat_offset
                angle_val = angle_tensor[current_idx]

                angle_deg = np.degrees(angle_val) if abs(angle_val) <= np.pi else angle_val

                raw_detections.append({
                    "box": ((float(cx_orig), float(cy_orig)), (float(w_orig), float(h_orig)), float(angle_deg)),
                    "score": float(score),
                    "class_id": int(cls_id)
                })

            global_box_idx += height * width

        if not raw_detections:
            return []

        r_boxes = [item["box"] for item in raw_detections]
        scores = [item["score"] for item in raw_detections]

        indices = cv2.dnn.NMSBoxesRotated(
            r_boxes, scores, score_threshold=score_thresh, nms_threshold=nms_thresh
        )

        if len(indices) == 0:
            return []

        indices = np.array(indices).flatten()
        return [raw_detections[idx] for idx in indices]

    def run(self, image_path, output_path="output_final.jpg"):
        if self.rknn.load_rknn(self.model_path) != 0:
            print("RKNN 모델 로드 실패")
            return

        try:
            self.rknn.init_runtime(core_mask=RKNN.NPU_CORE_0)
        except AttributeError:
            self.rknn.init_runtime()

        # 전처리 메서드 호출
        input_data, ratio, orig_img = self.preprocess(image_path)

        outputs = self.rknn.inference(inputs=[input_data])

        results = self.decode_outputs(
            outputs, ratio=ratio, score_thresh=0.45, nms_thresh=0.25
        )

        print(f"최종 감지 객체 개수: {len(results)}개")

        for det in results:
            box = det["box"]
            score = det["score"]
            class_id = det["class_id"]

            pts = cv2.boxPoints(box)
            pts = np.int32(pts)

            cv2.polylines(
                orig_img, [pts], isClosed=True, color=(0, 255, 0), thickness=2
            )

            label = f"ID:{class_id} {score:.2f}"
            cv2.putText(
                orig_img,
                label,
                (int(pts[0][0]), int(pts[0][1] - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 0, 255),
                1,
            )

        cv2.imwrite(output_path, orig_img)
        print(f"추론 완료: {output_path} 저장됨")

        self.rknn.release()

if __name__ == "__main__":
    detector = YoloOBBRKNN(model_path="./models/obb/yolo26n-obb-RK3588_640.rknn", input_size=640)
    detector.run(image_path="./images/obb.jpg")

W Query dynamic range failed. Ret code: RKNN_ERR_MODEL_INVALID. (If it is a static shape RKNN model, please ignore the above warning message.)


I RKNN: [13:50:43.099] RKNN Runtime Information, librknnrt version: 2.4.0 (b458df3b4a@2026-01-17T10:53:35)
I RKNN: [13:50:43.099] RKNN Driver Information, version: 0.9.8
I RKNN: [13:50:43.100] RKNN Model Information, version: 6, toolkit version: 2.3.2(compiler version: 2.3.2 (@2025-04-03T08:26:16)), target: RKNPU v2, target platform: rk3588, framework name: ONNX, framework layout: NCHW, model inference type: static_shape
W RKNN: [13:50:43.127] query RKNN_QUERY_INPUT_DYNAMIC_RANGE error, rknn model is static shape type, please export rknn with dynamic_shapes
최종 감지 객체 개수: 152개
추론 완료: output_final.jpg 저장됨
